# Aff-Wild2 Stage 2 — audio→video alignment

Linear-interpolates each per-video audio cache onto the visual frame timeline so the two modalities are co-indexed by `(videoname, frame_idx)`. After this notebook, every Stage 3 fusion variant can index a single batch element via `(visual_features[i], audio_features[i])`.

Output dirs:
* `cache/features/hubert_large_aligned/`
* `cache/features/wav2vec2_base_aligned/`

Each file contains `features (T_video, D_a)` and `has_audio (bool)`. `T_video` is read from the matching enet visual cache. Wall-time: ~5 min per encoder.

In [1]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

VISUAL_CACHE   = REPO / 'cache' / 'features' / 'enet_b0_8_va_mtl'
HUBERT_RAW     = REPO / 'cache' / 'features' / 'hubert_large'
WAV2VEC2_RAW   = REPO / 'cache' / 'features' / 'wav2vec2_base'
HUBERT_ALIGNED   = REPO / 'cache' / 'features' / 'hubert_large_aligned'
WAV2VEC2_ALIGNED = REPO / 'cache' / 'features' / 'wav2vec2_base_aligned'

for d in (VISUAL_CACHE, HUBERT_RAW, WAV2VEC2_RAW):
    assert d.exists(), f'missing input cache: {d}'
print('cwd          :', Path.cwd())
print('visual cache :', VISUAL_CACHE, '(', len(list(VISUAL_CACHE.glob('*.npz'))), 'files)')
print('hubert raw   :', HUBERT_RAW,   '(', len(list(HUBERT_RAW.glob('*.npz'))),   'files)')
print('wav2vec2 raw :', WAV2VEC2_RAW, '(', len(list(WAV2VEC2_RAW.glob('*.npz'))), 'files)')

cwd          : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code
visual cache : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features\enet_b0_8_va_mtl ( 307 files)
hubert raw   : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features\hubert_large ( 607 files)
wav2vec2 raw : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features\wav2vec2_base ( 607 files)


## 1. HuBERT-large × enet visual cache → `hubert_large_aligned`

For every visual `.npz`, looks up the matching audio `.npz` by stem and resamples its `(T_audio, 1024)` features onto a `(T_video, 1024)` grid. Missing audio is zero-filled with `has_audio=False`.

In [2]:
from src.features.align_audio_to_video import align_caches

HUBERT_ALIGNED.mkdir(parents=True, exist_ok=True)
align_caches(
    audio_cache_dir=HUBERT_RAW,
    visual_cache_dir=VISUAL_CACHE,
    output_dir=HUBERT_ALIGNED,
    fallback_dim=1024,
)
print(f'aligned files: {len(list(HUBERT_ALIGNED.glob("*.npz")))} -> {HUBERT_ALIGNED}')

aligned files: 307 -> C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features\hubert_large_aligned


In [3]:
import numpy as np

files = sorted(HUBERT_ALIGNED.glob('*.npz'))
assert len(files) == len(list(VISUAL_CACHE.glob('*.npz'))), \
    'aligned cache size != visual cache size'

has_audio = []
shape_mismatch = []
for fp in files:
    aligned = np.load(fp)
    visual  = np.load(VISUAL_CACHE / f'{fp.stem}.npz')
    if aligned['features'].shape[0] != visual['features'].shape[0]:
        shape_mismatch.append((fp.stem, aligned['features'].shape[0], visual['features'].shape[0]))
    has_audio.append(bool(aligned['has_audio']))
    assert aligned['features'].shape[1] == 1024

print(f'aligned files          : {len(files)}')
print(f'has_audio=True         : {sum(has_audio)} / {len(has_audio)}')
print(f'has_audio=False (zero) : {len(has_audio) - sum(has_audio)}')
print(f'T_audio != T_video     : {len(shape_mismatch)} (must be 0)')
if shape_mismatch:
    print('  sample mismatches:', shape_mismatch[:5])
    raise AssertionError('alignment shape mismatch \u2014 inspect the underlying caches')

aligned files          : 307
has_audio=True         : 307 / 307
has_audio=False (zero) : 0
T_audio != T_video     : 0 (must be 0)


## 2. wav2vec 2.0 base × enet visual cache → `wav2vec2_base_aligned`

In [4]:
WAV2VEC2_ALIGNED.mkdir(parents=True, exist_ok=True)
align_caches(
    audio_cache_dir=WAV2VEC2_RAW,
    visual_cache_dir=VISUAL_CACHE,
    output_dir=WAV2VEC2_ALIGNED,
    fallback_dim=768,
)
print(f'aligned files: {len(list(WAV2VEC2_ALIGNED.glob("*.npz")))} -> {WAV2VEC2_ALIGNED}')

aligned files: 307 -> C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features\wav2vec2_base_aligned


In [5]:
files = sorted(WAV2VEC2_ALIGNED.glob('*.npz'))
assert len(files) == len(list(VISUAL_CACHE.glob('*.npz')))

has_audio = []
for fp in files:
    aligned = np.load(fp)
    visual  = np.load(VISUAL_CACHE / f'{fp.stem}.npz')
    assert aligned['features'].shape[0] == visual['features'].shape[0]
    assert aligned['features'].shape[1] == 768
    has_audio.append(bool(aligned['has_audio']))

print(f'aligned files          : {len(files)}')
print(f'has_audio=True         : {sum(has_audio)} / {len(has_audio)}')
print(f'has_audio=False (zero) : {len(has_audio) - sum(has_audio)}')

aligned files          : 307
has_audio=True         : 307 / 307
has_audio=False (zero) : 0


Audio is now co-indexed with the visual cache. Proceed to `aw2_05_stage3_fusion.ipynb` to train the F0–F5 fusion variants.